# GAIA TELESCOPE — Intelligent Query Pipeline
Guillem Masdemont, Plabon Shaha, Pietro Sestito

## 1 · Environment setup & imports

In [36]:
# ── SSL fix for ARNES cluster ─────────────────────────────────────────────────
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import os
os.environ.setdefault('CURL_CA_BUNDLE', '')
os.environ.setdefault('REQUESTS_CA_BUNDLE', '')

# ── Standard imports ──────────────────────────────────────────────────────────
import json
import re
import time
import numpy as np
import pandas as pd
from IPython.display import display

print('Imports OK')

Imports OK


## 2 · HuggingFace cache & SSL

In [37]:
HF_CACHE = os.path.abspath('../hf_cache')
os.environ['HF_HOME']            = HF_CACHE
os.environ['HF_HUB_CACHE']       = os.path.join(HF_CACHE, 'hub')
os.environ['HF_HUB_DISABLE_XET'] = '1'

ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings()
os.environ['CURL_CA_BUNDLE']     = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''

print(f'HF cache: {HF_CACHE}')

HF cache: /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache


## 3 · Load vLLM model

In [38]:
from vllm import LLM, SamplingParams

MODEL_ID             = 'Qwen/Qwen2.5-7B-Instruct'
TENSOR_PARALLEL_SIZE = int(os.environ.get('TENSOR_PARALLEL_SIZE', '1'))
DTYPE                = 'float16'

if 'llm' not in globals():
    print(f'Loading model from {HF_CACHE} …')
    llm = LLM(
        model=MODEL_ID,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        dtype=DTYPE,
        gpu_memory_utilization=0.90,
        trust_remote_code=True,
    )
    print('Model loaded.')
else:
    print('Model already loaded — reusing.')

SAMPLING_PARAMS       = SamplingParams(temperature=0.0, max_tokens=512)
JUDGE_SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=512)

Model already loaded — reusing.


## 4 · LLM query parser and ADQL builder

In [95]:
# ── Valid Gaia DR3 columns the LLM may choose ─────────────────────────────────
VALID_COLUMNS = {
    'ra', 'dec', 'source_id', 'parallax', 'parallax_error',
    'pmra', 'pmdec', 'radial_velocity',
    'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp',
    'teff_gspphot', 'logg_gspphot', 'mh_gspphot',
    'phot_variable_flag', 'ruwe',
}

# ── Gaia DR3 internal tables available for crossmatch ────────────────────────
VALID_JOIN_TABLES = {
    'astrophysical_parameters',
    'vari_classifier_result',
    'vari_rrlyrae',
    'vari_cepheid',
    'vari_eclipsing_binary',
    'vari_rotation_modulation',
}

# ── Valid intents ─────────────────────────────────────────────────────────────
VALID_INTENTS = {
    'cone_search',          # Stars in a circular sky region
    'color_histogram',      # Colour distribution in a region
    'hr_diagram',           # Hertzsprung-Russell diagram (forces parallax+G+bp_rp)
    'stellar_population',   # Filter by star type (teff, logg, metallicity) in a region
    'variability_search',   # Variable stars in a region (phot_variable_flag, no JOIN)
    'internal_crossmatch',  # Join gaia_source with another DR3 table in a region
    'nearest_neighbours',   # Closest stars to the Sun (sky-wide, ORDER BY parallax)
    'velocity_computation', # 3D kinematics: pm + radial_velocity (cone optional)
}

# ── Column aliases the LLM might use ─────────────────────────────────────────
_COLUMN_ALIASES = {
    'magnitude':         'phot_g_mean_mag',
    'g_magnitude':       'phot_g_mean_mag',
    'g_mag':             'phot_g_mean_mag',
    'bp_magnitude':      'phot_bp_mean_mag',
    'rp_magnitude':      'phot_rp_mean_mag',
    'color':             'bp_rp',
    'colour':            'bp_rp',
    'temperature':       'teff_gspphot',
    'teff':              'teff_gspphot',
    'logg':              'logg_gspphot',
    'metallicity':       'mh_gspphot',
    'proper_motion_ra':  'pmra',
    'proper_motion_dec': 'pmdec',
    'variability':       'phot_variable_flag',
}

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an astronomy query parser for the Gaia DR3 database.

Convert the user query into a single JSON object. Return ONLY valid JSON — no markdown, no explanation.

JSON schema:
{
  "intent":     string,
  "ra":         float or null,
  "dec":        float or null,
  "radius":     float or null,
  "columns":    [list of strings],
  "filters":    {dict of column: value or {min, max}},
  "join_table": string or null,
  "limit":      int
}

Supported intents:
  cone_search          — stars in a circular sky region (ra, dec, radius required)
  color_histogram      — colour distribution in a region (ra, dec, radius required)
  hr_diagram           — HR diagram; cone optional for sky-wide (parallax+G+bp_rp always added)
  stellar_population   — filter by star type; cone optional for sky-wide queries
  variability_search   — variable stars in a region (ra, dec, radius required)
  internal_crossmatch  — join with another DR3 table; cone optional (join_table required)
  nearest_neighbours   — closest stars to the Sun, sky-wide (ra/dec/radius may be null)
  velocity_computation — 3D kinematics: proper motion + radial velocity (ra/dec/radius optional)

Valid column names:
  ra, dec, source_id, parallax, parallax_error,
  pmra, pmdec, radial_velocity,
  phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, bp_rp,
  teff_gspphot, logg_gspphot, mh_gspphot,
  phot_variable_flag, ruwe

Valid join_table values (internal_crossmatch only):
  astrophysical_parameters, vari_classifier_result, vari_rrlyrae,
  vari_cepheid, vari_eclipsing_binary, vari_rotation_modulation

Rules:
  - If a target name is given (e.g. 'Pleiades', 'Andromeda'), infer ra/dec from your knowledge.
  - If coordinates are unknown and intent requires them, use galactic centre: ra=266.4, dec=-29.0
  - Default radius: 1.0 degree
  - Default limit: 1000
  - ra/dec/radius may be null for nearest_neighbours, velocity_computation, hr_diagram, stellar_population, internal_crossmatch
  - ra/dec/radius are REQUIRED for cone_search, color_histogram, variability_search
  - join_table is null unless intent is internal_crossmatch
  - filters: use {"min": x, "max": y} for ranges, or a single value for equality
  - "brightest", "faintest", "hottest", "coolest", "fastest", "nearest" — 
  these are sky-wide superlatives, never use cone_search for them.
  Use stellar_population (brightness/temperature) or nearest_neighbours (distance).

Examples:

Query: "Show stars around the Pleiades"
{"intent":"cone_search","ra":56.75,"dec":24.12,"radius":1.0,
 "columns":["source_id","ra","dec","phot_g_mean_mag"],
 "filters":{},"join_table":null,"limit":1000}

Query: "HR diagram of stars near Omega Centauri"
{"intent":"hr_diagram","ra":201.7,"dec":-47.5,"radius":1.0,
 "columns":["source_id","parallax","phot_g_mean_mag","bp_rp"],
 "filters":{"parallax":{"min":0.1}},"join_table":null,"limit":2000}

Query: "Show me the 50 nearest stars"
{"intent":"nearest_neighbours","ra":null,"dec":null,"radius":null,
 "columns":["source_id","ra","dec","parallax","phot_g_mean_mag"],
 "filters":{},"join_table":null,"limit":50}

Query: "Red dwarfs near Barnard's star"
{"intent":"stellar_population","ra":269.45,"dec":4.69,"radius":2.0,
 "columns":["source_id","ra","dec","teff_gspphot","logg_gspphot","bp_rp"],
 "filters":{"teff_gspphot":{"min":2400,"max":3900},"logg_gspphot":{"min":4.5}},
 "join_table":null,"limit":1000}

Query: "Variable stars near the galactic centre"
{"intent":"variability_search","ra":266.4,"dec":-29.0,"radius":1.0,
 "columns":["source_id","ra","dec","phot_g_mean_mag","phot_variable_flag"],
 "filters":{"phot_variable_flag":"VARIABLE"},"join_table":null,"limit":1000}

Query: "How fast do stars near Orion move?"
{"intent":"velocity_computation","ra":83.8,"dec":-5.4,"radius":2.0,
 "columns":["source_id","ra","dec","pmra","pmdec","radial_velocity","parallax"],
 "filters":{"radial_velocity":{"min":-500,"max":500}},"join_table":null,"limit":1000}

Query: "Colour histogram of stars near Andromeda"
{"intent":"color_histogram","ra":10.68,"dec":41.27,"radius":0.5,
 "columns":["source_id","bp_rp","phot_g_mean_mag","phot_bp_mean_mag","phot_rp_mean_mag"],
 "filters":{"bp_rp":{"min":-1.0,"max":6.0}},"join_table":null,"limit":1000}

Query: "Show me the hottest stars in the whole sky"
{"intent":"stellar_population","ra":null,"dec":null,"radius":null,
 "columns":["source_id","ra","dec","teff_gspphot","phot_g_mean_mag"],
 "filters":{"teff_gspphot":{"min":30000}},"join_table":null,"limit":1000}

Query: "Cross-match stars near the LMC with astrophysical parameters"
{"intent":"internal_crossmatch","ra":80.9,"dec":-69.8,"radius":2.0,
 "columns":["source_id","ra","dec","teff_gspphot","logg_gspphot"],
 "filters":{},"join_table":"astrophysical_parameters","limit":1000}

Query: "Show me the brightest stars in the sky"
{"intent":"stellar_population","ra":null,"dec":null,"radius":null,
 "columns":["source_id","ra","dec","phot_g_mean_mag","bp_rp","parallax"],
 "filters":{"phot_g_mean_mag":{"max":4.0}},
 "join_table":null,"limit":1000}
"""


def _build_prompt(user_query: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{user_query}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def parse_query(user_query: str) -> dict:
    """Send query to LLM and return parsed JSON dict."""
    prompt  = _build_prompt(user_query)
    outputs = llm.generate([prompt], SAMPLING_PARAMS)
    raw     = outputs[0].outputs[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^```\s*',     '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'\s*```$',     '', raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f'Model returned non-JSON:\n{raw}') from exc


## 5 · ADQL builder

In [94]:
def _resolve_columns(cols: list) -> list:
    """Resolve aliases and deduplicate column list."""
    resolved = [_COLUMN_ALIASES.get(c, c) for c in cols]
    seen, out = set(), []
    for c in resolved:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out


def _cone_clause(ra, dec, radius, ra_col='ra', dec_col='dec') -> str:
    return f"1=CONTAINS(POINT('ICRS', {ra_col}, {dec_col}), CIRCLE('ICRS', {ra}, {dec}, {radius}))"


def _filter_clauses(filters: dict) -> list:
    """
    Convert filters dict to list of ADQL WHERE fragments.
    Supports:
      {"col": value}               -> col = value  (or col = 'value' for strings)
      {"col": {"min": x}}          -> col >= x
      {"col": {"max": y}}          -> col <= y
      {"col": {"min": x, "max": y}} -> col BETWEEN x AND y
    """
    clauses = []
    for col, val in filters.items():
        col = _COLUMN_ALIASES.get(col, col)
        if isinstance(val, dict):
            mn, mx = val.get('min'), val.get('max')
            if mn is not None and mx is not None:
                clauses.append(f'{col} BETWEEN {mn} AND {mx}')
            elif mn is not None:
                clauses.append(f'{col} >= {mn}')
            elif mx is not None:
                clauses.append(f'{col} <= {mx}')
        elif isinstance(val, str):
            clauses.append(f"{col} = '{val}'")
        else:
            clauses.append(f'{col} = {val}')
    return clauses


def _null_guards(columns: list) -> list:
    """
    For GSP-Phot columns, inject IS NOT NULL guards
    so the query doesn't silently return rows with missing data.
    """
    gspphot = {'teff_gspphot', 'logg_gspphot', 'mh_gspphot'}
    return [f'{c} IS NOT NULL' for c in columns if c in gspphot]


def build_adql(q: dict) -> str:
    """
    Convert a validated parsed JSON dict into an ADQL query string.

    Intents:
      cone_search         — spatial cone, LLM-chosen columns
      color_histogram     — cone (required), forces all 4 photometry columns
      hr_diagram          — cone optional, forces parallax + G + bp_rp, adds abs mag
      stellar_population  — cone optional, teff/logg/mh filters + null guards
      variability_search  — cone (required), phot_variable_flag = 'VARIABLE'
      internal_crossmatch — cone optional, JOIN on source_id to another DR3 table
      nearest_neighbours  — sky-wide, ORDER BY parallax DESC
      velocity_computation— radial_velocity IS NOT NULL, cone optional
    """
    intent     = q['intent']
    ra         = q.get('ra')
    dec        = q.get('dec')
    radius     = q.get('radius', 1.0)
    limit      = q.get('limit', 1000)
    filters    = q.get('filters') or {}
    join_table = q.get('join_table')
    columns    = _resolve_columns(q.get('columns') or ['source_id', 'ra', 'dec'])

    # These are must include fields. 
    for col in ('source_id', 'ra', 'dec', 'phot_g_mean_mag', 'bp_rp', 'pmra', 'pmdec'):
        if col not in columns:
            columns.append(col)

    # ── Sky-wide flag (no cone given) ─────────────────────────────────────────
    sky_wide = ra is None or dec is None

    filter_parts = _filter_clauses(filters)

    # cone_search
    if intent == 'cone_search':
        cols  = ', '.join(columns)
        where = ' AND '.join([_cone_clause(ra, dec, radius)] + filter_parts)
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # color_histogram
    elif intent == 'color_histogram':
        forced = ['source_id', 'bp_rp', 'phot_g_mean_mag',
                  'phot_bp_mean_mag', 'phot_rp_mean_mag']
        cols   = ', '.join(_resolve_columns(forced + columns))
        where_parts = [
            _cone_clause(ra, dec, radius),
            'bp_rp IS NOT NULL',
        ] + filter_parts
        where = ' AND '.join(where_parts)
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # hr_diagram
    elif intent == 'hr_diagram':
        forced  = ['source_id', 'parallax', 'phot_g_mean_mag', 'bp_rp']
        cols    = _resolve_columns(forced + columns)
        select_cols  = ', '.join(cols)
        select_cols += ', (phot_g_mean_mag + 5*LOG10(parallax/100)) AS abs_g_mag'
        where_parts  = ['parallax > 0', 'bp_rp IS NOT NULL'] + filter_parts
        if not sky_wide:
            where_parts.insert(0, _cone_clause(ra, dec, radius))
        where = ' AND '.join(where_parts)
        return f'SELECT TOP {limit} {select_cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # stellar_population
    elif intent == 'stellar_population':
        cols        = ', '.join(columns)
        null_guards = _null_guards(columns)
        where_parts = null_guards + filter_parts
        if not sky_wide:
            where_parts.insert(0, _cone_clause(ra, dec, radius))
        where = ' AND '.join(where_parts) if where_parts else '1=1'
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # variability_search
    elif intent == 'variability_search':
        if 'phot_variable_flag' not in columns:
            columns.append('phot_variable_flag')
        cols        = ', '.join(columns)
        where_parts = [
            _cone_clause(ra, dec, radius),
            "phot_variable_flag = 'VARIABLE'",
        ] + [fp for fp in filter_parts
             if 'phot_variable_flag' not in fp]
        where = ' AND '.join(where_parts)
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # internal_crossmatch 
    elif intent == 'internal_crossmatch':
        g_cols = ', '.join(f'g.{c}' if c in VALID_COLUMNS else c
                           for c in columns)
        where_parts = filter_parts[:]
        if not sky_wide:
            where_parts.insert(0, _cone_clause(ra, dec, radius, 'g.ra', 'g.dec'))
        where = ' AND '.join(where_parts) if where_parts else '1=1'
        return (
            f'SELECT TOP {limit} {g_cols} '
            f'FROM gaiadr3.gaia_source AS g '
            f'JOIN gaiadr3.{join_table} AS x ON g.source_id = x.source_id '
            f'WHERE {where} ORDER BY random_index'
        )

    # nearest_neighbours
    elif intent == 'nearest_neighbours':
        if 'parallax' not in columns:
            columns.insert(0, 'parallax')
        cols        = ', '.join(columns)
        where_parts = ['parallax > 0'] + filter_parts
        where       = ' AND '.join(where_parts)
        return (
            f'SELECT TOP {limit} {cols} '
            f'FROM gaiadr3.gaia_source '
            f'WHERE {where} '
            f'ORDER BY parallax DESC'
        )

    # velocity_computation 
    elif intent == 'velocity_computation':
        forced = ['source_id', 'ra', 'dec', 'pmra', 'pmdec', 'radial_velocity']
        cols   = ', '.join(_resolve_columns(forced + columns))
        where_parts = ['radial_velocity IS NOT NULL'] + filter_parts
        if not sky_wide:
            where_parts.insert(0, _cone_clause(ra, dec, radius))
        where = ' AND '.join(where_parts)
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    else:
        raise ValueError(f"Unknown intent: '{intent}'")

## 6 · Gaia TAP query runner

In [93]:
def run_query(adql: str, retries: int = 3, backoff: float = 15.0, row_limit: int = 50_000) -> pd.DataFrame:
    """
    Submit ADQL to the Gaia TAP service, return a pandas DataFrame.
    Injects TOP N if missing. Fails fast on HTTP 500 (bad syntax).
    """
    from astroquery.gaia import Gaia

    Gaia.MAIN_GAIA_TABLE = 'gaiadr3.gaia_source'
    Gaia.ROW_LIMIT       = row_limit

    if not re.search(r'\bTOP\s+\d+\b', adql, re.IGNORECASE):
        adql = re.sub(r'\bSELECT\b', f'SELECT TOP {row_limit}',
                      adql, count=1, flags=re.IGNORECASE)
        print(f'Injected TOP {row_limit}')

    if not re.search(r'\bWHERE\b', adql, re.IGNORECASE):
        print('⚠️  No WHERE clause — query may time out.')

    print(f'\nQuery sent:\n{adql}\n')

    last_exc = None
    for attempt in range(retries):
        try:
            job    = Gaia.launch_job_async(adql, verbose=False)
            result = job.get_results()
            print(f'Got {len(result):,} rows')
            return result.to_pandas()
        except Exception as exc:
            last_exc = exc
            print(f'[attempt {attempt+1}/{retries}] {exc}')
            if '500' in str(exc):
                print('HTTP 500 — bad query syntax, not retrying.')
                raise exc
            if attempt < retries - 1:
                wait = backoff * (2 ** attempt)
                print(f'Waiting {wait:.0f}s …')
                time.sleep(wait)
    raise last_exc

## 7 · Teacher — JSON & ADQL validator

In [83]:
MAX_RETRIES = 3

# Intents where cone is fully optional (sky-wide queries allowed)
_CONE_OPTIONAL_INTENTS = {
    'nearest_neighbours', 'velocity_computation',
    'hr_diagram', 'stellar_population', 'internal_crossmatch',
}
# Intents that always require a spatial anchor
_CONE_REQUIRED_INTENTS = {'cone_search', 'color_histogram', 'variability_search'}


def validate_parsed_json(q: dict) -> list:
    """Return list of error strings. Empty list means valid."""
    errors = []
    intent = q.get('intent')

    # ── intent ────────────────────────────────────────────────────────────────
    if not intent or intent not in VALID_INTENTS:
        errors.append(
            f"'intent' is '{intent}', must be one of: {sorted(VALID_INTENTS)}."
        )
        return errors  # no point checking the rest

    # ── spatial fields ────────────────────────────────────────────────────────
    needs_cone = intent in _CONE_REQUIRED_INTENTS
    for field in ('ra', 'dec'):
        val = q.get(field)
        if val is None:
            if needs_cone:
                errors.append(
                    f"'{field}' is null for intent '{intent}'. "
                    f"Provide coordinates or use galactic centre (ra=266.4, dec=-29.0)."
                )
        else:
            if not isinstance(val, (int, float)):
                errors.append(f"'{field}' must be a number, got: {repr(val)}.")
            elif field == 'ra' and not (0 <= float(val) <= 360):
                errors.append(f"'ra' = {val} is outside [0, 360].")
            elif field == 'dec' and not (-90 <= float(val) <= 90):
                errors.append(f"'dec' = {val} is outside [-90, 90].")

    radius = q.get('radius')
    if needs_cone:
        if radius is None:
            errors.append("'radius' is null. Use default 1.0 degree.")
        elif not isinstance(radius, (int, float)) or float(radius) <= 0:
            errors.append(f"'radius' must be positive, got: {repr(radius)}.")
        elif float(radius) > 10:
            errors.append(f"'radius' = {radius}° is very large (> 10°) and may time out.")

    # ── limit ─────────────────────────────────────────────────────────────────
    limit = q.get('limit', 1000)
    if not isinstance(limit, int) or limit <= 0:
        errors.append(f"'limit' must be a positive integer, got: {repr(limit)}.")
    elif limit > 100_000:
        errors.append(f"'limit' = {limit:,} is too large. Use ≤ 100,000.")

    # ── columns ───────────────────────────────────────────────────────────────
    cols    = q.get('columns') or []
    unknown = [c for c in cols
               if c not in VALID_COLUMNS and c not in _COLUMN_ALIASES]
    if unknown:
        errors.append(
            f"Unknown column(s): {unknown}. "
            f"Allowed: {sorted(VALID_COLUMNS)}."
        )

    # ── join_table ────────────────────────────────────────────────────────────
    join_table = q.get('join_table')
    if intent == 'internal_crossmatch':
        if not join_table:
            errors.append(
                f"'join_table' is required for internal_crossmatch. "
                f"Choose one of: {sorted(VALID_JOIN_TABLES)}."
            )
        elif join_table not in VALID_JOIN_TABLES:
            errors.append(
                f"'join_table' = '{join_table}' is not a valid DR3 table. "
                f"Choose one of: {sorted(VALID_JOIN_TABLES)}."
            )
    elif join_table is not None:
        errors.append(
            f"'join_table' should be null for intent '{intent}', got: '{join_table}'."
        )

    return errors


def validate_adql(adql: str) -> list:
    """Catch common problems in the final ADQL string."""
    errors = []
    if re.search(r'\bNone\b', adql):
        errors.append(
            "ADQL contains literal 'None' — a coordinate was not resolved. "
            "Replace with numeric degree values."
        )
    if not re.search(r'\bTOP\s+\d+\b', adql, re.IGNORECASE):
        errors.append('No TOP clause — add TOP N to cap result size.')
    if not re.search(r'\bWHERE\b', adql, re.IGNORECASE):
        errors.append('No WHERE clause — add a spatial or quality filter.')
    return errors


def build_correction_prompt(original_query: str, bad_json: dict,
                             errors: list, attempt: int) -> str:
    error_block = '\n'.join(f'  - {e}' for e in errors)
    return (
        f'Your previous answer for the query:\n'
        f'  "{original_query}"\n\n'
        f'produced this JSON:\n'
        f'{json.dumps(bad_json, indent=2)}\n\n'
        f'The teacher found {len(errors)} error(s) (attempt {attempt}/{MAX_RETRIES}):\n'
        f'{error_block}\n\n'
        f'Fix ALL errors and return ONLY corrected JSON. No markdown, no explanation.'
    )


print('Validator defined.')

Validator defined.


## 8 · Cost judge

In [96]:
COST_JUDGE_SYSTEM_PROMPT = """You are an expert in ADQL queries on the Gaia DR3 database (1.8 billion rows).
Evaluate how expensive a query will be on the Gaia TAP server.

Return ONLY valid JSON, no markdown:
{
  "verdict":       "cheap" | "moderate" | "expensive" | "dangerous",
  "score":         0-100,
  "reasons":       [list of strings],
  "optimisations": [list of concrete ADQL rewrites],
  "safe_to_run":   true | false
}

Scoring guide:
   0-25  cheap     : TOP ≤ 1000, tight cone ≤ 0.5°, indexed filter, no JOIN
  26-50  moderate  : TOP ≤ 10000, cone ≤ 2°, simple filters
  51-75  expensive : TOP ≤ 50000, cone > 2°, ORDER BY non-indexed col, weak parallax
  76-100 dangerous : no TOP or TOP > 100000, no spatial filter, full-table scan, multi-JOIN

Key cost drivers:
  - Missing/large TOP (biggest risk)
  - Cone radius > 2 degrees
  - ORDER BY on parallax or radial_velocity (not indexed)
  - radial_velocity IS NOT NULL (only ~1% of stars, forces full scan)
  - parallax > 0  (matches ~90% of rows — very weak)
  - JOIN without a tight spatial filter
  - No WHERE clause

Optimisation strategies:
  - Reduce TOP N
  - Shrink cone radius
  - Replace parallax > 0 with parallax > 10 (nearby stars)
  - Add phot_g_mean_mag < 15 to cut faint stars early
  - Remove ORDER BY or replace with a parallax threshold filter
  - For sky-wide queries with no CONTAINS clause: suggest MOD(random_index, N) = 0
    as a uniform sky sample instead of adding a spatial cone.
    N=100 for moderate, N=1000 for expensive, N=10000 for dangerous."""


def _build_cost_prompt(adql: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{COST_JUDGE_SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'Evaluate this ADQL query:\n\n{adql}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def fast_cost_precheck(adql: str):
    """Rule-based fast check — no LLM call. Returns dict if dangerous, else None."""
    adql_upper = adql.upper()
    if 'CONTAINS' not in adql_upper and 'WHERE' not in adql_upper:
        return {
            'verdict': 'dangerous', 'score': 100, 'safe_to_run': False,
            'reasons': ['No WHERE clause — full scan of 1.8B rows.'],
            'optimisations': ['Add MOD(random_index, 10000) = 0 for a uniform sky sample.'],
        }
    top_match = re.search(r'\bTOP\s+(\d+)\b', adql, re.IGNORECASE)
    if not top_match:
        return {
            'verdict': 'dangerous', 'score': 95, 'safe_to_run': False,
            'reasons': ['No TOP clause — unbounded result set.'],
            'optimisations': ['Add TOP 1000 after SELECT.'],
        }
    if int(top_match.group(1)) > 100_000:
        return {
            'verdict': 'dangerous', 'score': 90, 'safe_to_run': False,
            'reasons': [f'TOP {int(top_match.group(1)):,} exceeds 100,000 row limit.'],
            'optimisations': ['Reduce TOP to ≤ 10,000.'],
        }
    return None


def evaluate_query_cost(adql: str) -> dict:
    """LLM judge: score and explain query cost."""
    prompt  = _build_cost_prompt(adql)
    outputs = llm.generate([prompt], JUDGE_SAMPLING_PARAMS)
    raw     = outputs[0].outputs[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^```\s*',     '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'\s*```$',     '', raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f'Cost judge returned non-JSON:\n{raw}')
        return {
            'verdict': 'expensive', 'score': 70, 'safe_to_run': False,
            'reasons': ['Cost judge failed to parse — defaulting conservative.'],
            'optimisations': [],
        }


def auto_optimise_adql(adql: str, optimisations: list,
                       top_cap: int = 5_000, score: int = 0) -> str:
    """Apply deterministic optimisations to the ADQL string."""

    #Inject random sampling for sky-wide queries
    has_cone     = 'CONTAINS' in adql.upper()
    has_sampling = 'random_index' in adql.lower()
    if not has_cone and not has_sampling:
        if score >= 75:
            sample_n = 10000   # 0.01% — dangerous
        elif score >= 50:
            sample_n = 1000    # 0.1%  — expensive
        else:
            sample_n = 100     # 1%    — moderate
        adql = re.sub(
            r'\bWHERE\b',
            f'WHERE MOD(random_index, {sample_n}) = 0 AND',
            adql, count=1, flags=re.IGNORECASE
        )
        print(f'  Injected MOD(random_index, {sample_n}) — uniform sky sample')

    #Cap Top M 
    def cap_top(m):
        n = int(m.group(1))
        if n > top_cap:
            print(f'  TOP {n:,} → {top_cap:,}')
            return f'TOP {top_cap}'
        return m.group(0)
    adql = re.sub(r'\bTOP\s+(\d+)\b', cap_top, adql, flags=re.IGNORECASE)

    # ── Shrink large cones ────────────────────────────────────────────────────
    def shrink_cone(m):
        if float(m.group(3)) > 3.0:
            print(f'  Cone {m.group(3)}° → 2.0°')
            return f"CIRCLE('ICRS', {m.group(1)}, {m.group(2)}, 2.0)"
        return m.group(0)
    adql = re.sub(
        r"CIRCLE\s*\(\s*'ICRS'\s*,\s*([0-9.\-]+)\s*,\s*([0-9.\-]+)\s*,\s*([0-9.]+)\s*\)",
        shrink_cone, adql, flags=re.IGNORECASE
    )

    # ── Tighten weak parallax filter ──────────────────────────────────────────
    before = adql
    adql   = re.sub(r'\bparallax\s*>\s*0\b', 'parallax > 1', adql, flags=re.IGNORECASE)
    if adql != before:
        print('  parallax > 0  →  parallax > 1')

    if optimisations:
        print('  LLM judge also suggested:')
        for opt in optimisations:
            print(f'    • {opt}')
    return adql


_VERDICT_EMOJI = {'cheap': '🟢', 'moderate': '🟡', 'expensive': '🟠', 'dangerous': '🔴'}


def print_cost_report(cost: dict):
    verdict = cost.get('verdict', '?')
    score   = cost.get('score', '?')
    emoji   = _VERDICT_EMOJI.get(verdict, '⚪')
    print(f"  {'─'*52}")
    print(f"  {emoji}  COST: {verdict.upper()}  (score {score}/100)")
    print(f"  {'─'*52}")
    for r in cost.get('reasons', []):
        print(f'    ⚠️  {r}')
    for o in cost.get('optimisations', []):
        print(f'    💡 {o}')
    print(f"  {'─'*52}")


## 9 · Supervised pipeline

In [97]:
def supervised_pipeline(user_query: str) -> pd.DataFrame:
    """
    Full pipeline:
      1. LLM parses query → JSON
      2. Teacher validates JSON  (correction loop)
      3. build_adql()
      4. Teacher validates ADQL  (correction loop)
      5. Cost judge              (block / optimise / approve)
      6. run_query()             (retry on transient errors)

    Raises RuntimeError after MAX_RETRIES exhausted.
    """
    print(f"\n{'='*60}")
    print(f'  USER QUERY: {user_query}')
    print(f"{'='*60}\n")

    current_prompt = user_query
    parsed = None
    adql   = None

    for attempt in range(1, MAX_RETRIES + 1):

        #1: LLM to JSON 
        print(f'[Attempt {attempt}/{MAX_RETRIES}] Parsing …')
        try:
            parsed = parse_query(current_prompt)
        except ValueError as exc:
            print(f'  Non-JSON from LLM: {exc}')
            current_prompt = (
                f'Your answer was not valid JSON.\n'
                f'Original query: "{user_query}"\n'
                f'Error: {exc}\n'
                f'Return ONLY a valid JSON object.'
            )
            continue

        print(f'  Parsed JSON:\n{json.dumps(parsed, indent=2)}')

        #2: validate JSON
        json_errors = validate_parsed_json(parsed)
        if json_errors:
            print(f'  JSON errors ({len(json_errors)}):')
            for e in json_errors:
                print(f'    • {e}')
            current_prompt = build_correction_prompt(
                user_query, parsed, json_errors, attempt
            )
            continue
        print('  JSON valid.')

        #3: build ADQL
        try:
            adql = build_adql(parsed)
        except (ValueError, KeyError) as exc:
            print(f'  build_adql() failed: {exc}')
            current_prompt = build_correction_prompt(
                user_query, parsed, [f'build_adql() error: {exc}'], attempt
            )
            continue
        print(f'  ADQL: {adql}')

        #4: validate ADQL
        adql_errors = validate_adql(adql)
        if adql_errors:
            print(f'  ADQL errors ({len(adql_errors)}):')
            for e in adql_errors:
                print(f'    • {e}')
            current_prompt = build_correction_prompt(
                user_query, parsed, adql_errors, attempt
            )
            continue
        print('  ADQL syntax valid.')

        # 5: cost gate
        print('  Evaluating cost …')
        cost = fast_cost_precheck(adql)
        if cost is None:
            cost = evaluate_query_cost(adql)
        print_cost_report(cost)

        if cost['verdict'] == 'dangerous':
            print('  🔴 BLOCKED — query too dangerous.')
            current_prompt = build_correction_prompt(
                user_query, parsed,
                [f"Query rated DANGEROUS (score {cost['score']}/100). "
                 f"Reasons: {'; '.join(cost['reasons'])}. "
                 f"Fix: {'; '.join(cost['optimisations'])}"],
                attempt,
            )
            continue

        if not cost['safe_to_run']:
            adql = auto_optimise_adql(adql, cost.get('optimisations', []), score=cost.get('score', 0))
            print(f'  Optimised ADQL: {adql}')

        # 6: execute 
        try:
            return run_query(adql)
        except Exception as exc:
            print(f'  run_query() failed: {exc}')
            if '500' in str(exc):
                current_prompt = build_correction_prompt(
                    user_query, parsed,
                    [f'ADQL caused HTTP 500 on Gaia TAP: {exc}'],
                    attempt,
                )
                continue
            raise

    raise RuntimeError(
        f'Pipeline failed after {MAX_RETRIES} attempts for query: "{user_query}"'
    )

## 10 · Run a query

In [98]:
# Spatial
# USER_QUERY = 'Show me stars around the Pleiades'
# USER_QUERY = 'Stars near the galactic centre within 0.5 degrees'

# Photometry
#USER_QUERY = 'Colour histogram of stars near Andromeda'
#USER_QUERY = 'Show me the red dwarfs'

# HR diagram
# USER_QUERY = 'HR diagram of stars near Omega Centauri'

# Stellar population
# USER_QUERY = 'Show me metal-poor giant stars near the galactic centre'
#USER_QUERY = 'Show me metal-poor giant stars'

# Variability
# USER_QUERY = 'Variable stars near the LMC'

# Kinematics
#USER_QUERY = 'How fast do stars near Orion move?'
#USER_QUERY   = 'Show me the 50 nearest stars to the Sun'
#USER_QUERY = 'Pick the fastest stars' 


# Crossmatch
# USER_QUERY = 'Cross-match stars near the galactic centre with astrophysical parameters'


USER_QUERY = 'Provide me the brightest stars'

try:
    df = supervised_pipeline(USER_QUERY)
    display(df.head(10))
except RuntimeError as e:
    print(f'\n🔴  {e}')


  USER QUERY: Provide me the brightest stars

[Attempt 1/3] Parsing …


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.72s/it, est. speed input: 869.01 toks/s, output: 37.76 toks/s]


  Parsed JSON:
{
  "intent": "stellar_population",
  "ra": null,
  "dec": null,
  "radius": null,
  "columns": [
    "source_id",
    "ra",
    "dec",
    "phot_g_mean_mag",
    "bp_rp",
    "parallax"
  ],
  "filters": {
    "phot_g_mean_mag": {
      "max": 4.0
    }
  },
  "join_table": null,
  "limit": 1000
}
  JSON valid.
  ADQL: SELECT TOP 1000 source_id, ra, dec, phot_g_mean_mag, bp_rp, parallax, pmra, pmdec FROM gaiadr3.gaia_source WHERE phot_g_mean_mag <= 4.0 ORDER BY random_index
  ADQL syntax valid.
  Evaluating cost …


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.97s/it, est. speed input: 280.55 toks/s, output: 43.63 toks/s]


  ────────────────────────────────────────────────────
  🟢  COST: CHEAP  (score 20/100)
  ────────────────────────────────────────────────────
    ⚠️  TOP ≤ 1000
    ⚠️  No spatial filter
    ⚠️  ORDER BY on random_index (indexed)
    ⚠️  No JOIN
    💡 No further optimisations needed
  ────────────────────────────────────────────────────

Query sent:
SELECT TOP 1000 source_id, ra, dec, phot_g_mean_mag, bp_rp, parallax, pmra, pmdec FROM gaiadr3.gaia_source WHERE phot_g_mean_mag <= 4.0 ORDER BY random_index

INFO: Query finished. [astroquery.utils.tap.core]
Got 634 rows


,source_id,ra,dec,phot_g_mean_mag,bp_rp,parallax,pmra,pmdec
0,5614471482318442240,117.323539,-24.859795,3.023800,1.329295,3.005899,-3.610000,-2.579391
1,816649002967779584,135.157358,41.781756,3.959608,0.613851,53.361750,-494.613183,-285.415566
2,1193030490492925824,239.114697,15.655913,3.694426,0.703480,89.564664,311.182547,-1282.767037
3,1010770873228612608,135.906171,47.156273,3.798615,0.077129,NaN,NaN,NaN
4,1144716265942885888,144.271550,81.326307,3.785223,1.618635,3.542272,-16.340229,-16.607712
5,5330160246631998848,136.038353,-47.097775,3.418618,1.377886,10.826529,-47.861823,-8.972800
6,2962546605447869184,76.365384,-22.371368,2.653493,1.549777,15.599908,24.048504,-74.950974
7,546439965996109696,30.858140,72.421394,3.948435,0.056805,20.839131,-43.842376,22.547917
8,6319542659461082752,229.251282,-9.383003,2.634658,0.349665,NaN,NaN,NaN
9,1186417202929896960,221.524378,15.131856,3.827463,3.164297,5.108427,-83.147410,16.911453


## 11 · Report

In [92]:
import importlib
import display_html
importlib.reload(display_html)
from display_html import generate_report

generate_report(df, USER_QUERY, output_path='gaia_report.html')

Building report ...
Report saved: gaia_report.html  (1001 KB)
